In [27]:
import pandas as pd
import numpy as np

In [28]:
customers_cols = ['id']
products_cols = ['id','name']
product_variants_cols = ['id','product_id']
orders_cols = ['id', 'customer_id', 'status']
order_items_cols = ['id','order_id','product_variant_id']

In [29]:
customers = pd.read_csv("./dados/customers.csv", usecols=customers_cols)
orders = pd.read_csv("./dados/orders.csv", usecols=orders_cols)
order_items = pd.read_csv("./dados/order_items.csv", usecols=order_items_cols)
products = pd.read_csv("./dados/products.csv", usecols=products_cols)
product_variants = pd.read_csv("./dados/product_variants.csv", usecols=product_variants_cols)

### Criação da matriz

Importei os arquivos _products.csv_ e _customers.csv_ para saber quantos clientes existem na base de dados, qual é o tamanho esperado da matriz e obter o ID do produto referência da análise (uma vez que conheço apenas o seu nome).

Caso o cadastro de um cliente na base não esteja vinculado ao ato de compra, seria possível ter clientes que não aparecem na base de dados de pedidos (depende da regra de negócio). A lógica é a seguinte: ao conferir a tabela orders e ver que a quantidade de IDs únicos de cliente é igual à quantidade de entradas na tabela customers, sabemos que todos os clientes cadastrados na base de dados de clientes compraram algo, portanto partir da base orders para fazer a junção não incorrerá em viés de sobrevivência.

In [30]:
np.unique(orders["customer_id"]).shape

(2000,)

In [31]:
customers.shape

(2000, 1)

In [32]:
products.shape

(500, 2)

Teremos uma matriz de 2000 linhas e 500 colunas. Confirmamos que é possível começar a partir de orders para unir os dados, porém, essa verificação não confirma indubitavelmente qual é a regra de negócio (pode ser coincidência), o que significa que é possível no futuro ter clientes na tabela customers que não compraram nada e por isso não seriam representados na matriz. Desse modo, a decisão mais cautelosa e longeva é começar a junção a partir da tabela customers. 

In [33]:
customers.rename(columns={'id': 'customer_id'}, inplace=True)
df = pd.merge(customers, orders, on="customer_id", how="left")
df.shape

(48998, 3)

In [34]:
df.rename(columns={'id': 'order_id'}, inplace=True)
df = pd.merge(df, order_items, on="order_id")
df.shape

(147320, 5)

In [35]:
product_variants.rename(columns={'id': 'product_variant_id'}, inplace=True)
df = pd.merge(df, product_variants, on="product_variant_id")
df.shape

(147320, 6)

In [36]:
df

,customer_id,order_id,status,id,product_variant_id,product_id
0,1,4577,paid,13860,91,47
1,1,4577,paid,13861,978,484
2,1,4577,paid,13862,606,297
3,1,5220,confirmed,15856,859,426
4,1,5220,confirmed,15857,234,115
...,...,...,...,...,...,...
147315,2000,42295,paid,127296,175,86
147316,2000,42295,paid,127297,798,392
147317,2000,45030,paid,135412,383,190
147318,2000,46340,paid,139358,416,205


In [37]:
# Filtrar por pedidos confirmados ou pagos
df = df[df["status"].isin(["paid", "confirmed"])]
print(np.unique(df["status"]))
print(df.shape)

['confirmed' 'paid']
(125496, 6)


In [38]:
matriz = pd.crosstab(df["customer_id"], df["product_id"])
matriz

product_id,1,2,3,4,5,6,7,8,9,10,...,491,492,493,494,495,496,497,498,499,500
customer_id,,,,,,,,,,,,,,,,,,,,,
1,0,1,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,1,0,...,0,0,1,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,1,0,0,...,0,0,0,0,1,0,0,0,0,0
5,0,0,1,0,0,1,0,0,1,0,...,0,1,0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1996,0,0,0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1998,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,1,0,0,0


### Cálculo de Similaridade entre Produtos

In [39]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculando a similaridade de cosseno produto × produto 
# (o scikit-learn por padrão calcula a similaridade entre as linhas, portanto foi necessário transpor a matriz)
similaridade = cosine_similarity(matriz.T)
similaridade

array([[1.        , 0.18115377, 0.14275243, ..., 0.17101749, 0.07091217,
        0.12110326],
       [0.18115377, 1.        , 0.13548326, ..., 0.15334889, 0.11160714,
        0.11488092],
       [0.14275243, 0.13548326, 1.        , ..., 0.16489778, 0.07566884,
        0.1364712 ],
       ...,
       [0.17101749, 0.15334889, 0.16489778, ..., 1.        , 0.06492594,
        0.15422404],
       [0.07091217, 0.11160714, 0.07566884, ..., 0.06492594, 1.        ,
        0.13402774],
       [0.12110326, 0.11488092, 0.1364712 , ..., 0.15422404, 0.13402774,
        1.        ]], shape=(500, 500))

### Ranking de produtos similares

In [40]:
# Obter ID do produto referência
id = products[products["name"] == "Motor de Popa 1949"]["id"].iloc[0]
print(id)

180


In [41]:
arr_produto = similaridade[id] # ID começa a partir de 1

In [42]:
# O próprio motor deve ser desconsiderado no ranking
arr_produto = np.delete(arr_produto, id, axis=None)
arr_produto.shape

(499,)

In [ ]:
def top_k_compativel(arr, k, axis=-1):
    # Faz uma partição eficiente para isolar os maiores elementos no final
    partitioned_ind = np.argpartition(arr, -k, axis=axis)
    # Pega apenas os k últimos índices dessa partição
    topk_ind = np.take(partitioned_ind, range(-k, 0), axis=axis)
    # Extrai os valores correspondentes
    topk_vals = np.take_along_axis(arr, topk_ind, axis=axis)
    # Ordena os resultados de forma decrescente
    sort_ind = np.argsort(-topk_vals, axis=axis)
    final_ind = np.take_along_axis(topk_ind, sort_ind, axis=axis)
    final_vals = np.take_along_axis(topk_vals, sort_ind, axis=axis)
    
    return final_vals, final_ind

valores, indices = top_k_compativel(arr_produto, k=5)

In [44]:
print(valores, indices)

[0.20349339 0.19513112 0.18488264 0.18163774 0.17964522] [247  54 331 157  75]


In [ ]:
# Filtra e ordena o dataset de produtos de acordo com a lista 'indices'
products.set_index("id").loc[indices]

,name
id,
247,Defensa Náutica 7459
54,Colete Salva-Vidas 6302
331,Bateria Náutica 5936
157,Bússola de Bordo 1248
75,Vela Mestra 1913


### Validação

**Questão 7.2 - Validação**
Qual é o nome do produto com MAIOR similaridade ao “Motor de Popa 1949”?

Defensa Náutica 7459.

**Questão 7.3 - Explique:**
- Como a matriz foi construída?

    O primeiro passo foi verificar a quantidade de clientes e de produtos para saber qual seria a dimensão da matriz. Em seguida, fiz uma operação de merge entre os datasets customers, orders, order_items e product_variants (nessa ordem) para obter a relação entre cliente e produto - optei por começar de customers para garantir que todo cliente tenha uma linha correspondente na matriz, mesmo aqueles que nunca compraram produto algum (regra de negócio de cadastro de clientes no banco de dados é desconhecida). Filtrei os dados para contemplar apenas pedidos que foram confirmados ou pagos, a fim de refletir casos em que houve verdadeira intenção de compra. Por fim, usei a função crosstab da biblioteca pandas com id_cliente e id_produto como parâmetros para criar uma matriz em que cada linha representa um cliente e cada coluna um produto.
- O que significa a similaridade de cosseno nesse contexto?
    
    "A similaridade de cosseno é uma métrica matemática que mede o cosseno do ângulo entre dois vetores em um espaço multidimensional. Ela avalia a direção dos vetores e ignora a magnitude (o tamanho) deles."
    
    Ao calcular a similaridade de cosseno produto a produto, descobrimos quais produtos costumam ser comprados juntos ou possuem o mesmo padrão de público. Em contextos nos quais existem apenas valores positivos (como é o caso - temos uma matriz de compras positiva, onde o cliente comprou ou não), o resultado varia de 0 a 1. Quanto maior o valor de uma célula ij, maior é o grau de associação ou semelhança no padrão de consumo entre os produtos i e j.

- Uma limitação desse método de recomendação.

    Esse método informa que existe uma associação entre dois produtos, mas não informa qual a probabilidade de eles serem comprados juntos nem revela o perfil do cliente que costuma comprar os itens. Em suma, a similaridade de cosseno responde o que é parecido, mas não por que ou com qual frequência probabilística a ação vai acontecer.
